In [5]:
# LOADING THE DOCUMENT

from langchain_community.document_loaders import TextLoader
loaded_doc = TextLoader("speech.txt")
doc = loaded_doc.load()


In [6]:
# SPLITTING THE DOCUMENT

from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 800,
    chunk_overlap = 50
)

chunks = splitter.split_documents(doc)

In [ ]:
# CREATE EMBEDDINGS OF THE CHUNKS USING OLLAMA EMBEDDINGS

from langchain_community.embeddings import OllamaEmbeddings
embeddings = OllamaEmbeddings(model = "nomic-embed-text")
vectors = embeddings.embed_documents(chunks)

# THERE ARE 2 VECTOR DBs DISCUSSED BELLOW FAISS AND CROMA

### FAISS:

In [12]:
# STORE THE EMBEDDINGS INTO FIASS VECTORE DATABASE

from langchain_community.vectorstores import FAISS

# Store embeddings in FAISS
vectorstore = FAISS.from_documents(chunks, embeddings)


### CHROMA:

In [ ]:
from langchain_chroma import Chroma

# NOTE: WE CAN QUERY A VECTOR DB EVEN WITHOUT SAVING IT LOCALLY

# Create and Save Chroma DB locally (persist to disk)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="my_collection",
    persist_directory="./chroma_db"   # folder where DB is saved, IF WE DONNOT WANT TO SAVE THAN JUST SIMPLY REMOVE THIS LINE
)

# Load Chroma DB from disk,
vectorstore = Chroma(
    collection_name="my_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_db"
)

# NOTE: WE WILL QUERY THE DB SIMILARLY WE DO IN THE FAISS DB SAME 3 WAYS

# BELOW ARE THE WAYS OF QUERING THE VECTORE DB

#### WAY 1:

In [15]:
#  Similarity search (plain), Returns the top-k most similar Document objects
results = vectorstore.similarity_search("According to the essay, does having more choices always make people happier?", k=3)

for doc in results:
    print(doc.page_content)
    # print(doc)
    print("---")


The Paradox of Choice: When More is Less
In modern consumer societies, we are raised on the mantra that "more is better." We assume that an abundance of options—whether in the grocery aisle, the job market, or on streaming platforms—is the ultimate expression of freedom. However, psychologist Barry Schwartz argues the opposite: that an excess of choice actually leads to anxiety, paralysis, and a decreased sense of well-being.
---
The first hurdle of "choice overload" is analysis paralysis. When faced with dozens of similar options, the mental effort required to compare them becomes exhausting. Instead of making a "better" choice, many people simply walk away or defer the decision indefinitely. Even when a choice is finally made, a second issue arises: opportunity cost. By picking one path, we become hyper-aware of the attractive features of the paths we rejected. This leads to a lingering sense of regret, even if our chosen option is perfectly functional.
---
Furthermore, high expectat

#### WAY 2

In [14]:
# Similarity search with scores, Returns (Document, score) tuples — lower score = more similar (it's L2 distance).

results = vectorstore.similarity_search_with_score("According to the essay, does having more choices always make people happier?", k=3)

for doc, score in results:
    print(f"Score: {score:.4f}")
    print(doc.page_content)
    print("---")

Score: 243.3997
The Paradox of Choice: When More is Less
In modern consumer societies, we are raised on the mantra that "more is better." We assume that an abundance of options—whether in the grocery aisle, the job market, or on streaming platforms—is the ultimate expression of freedom. However, psychologist Barry Schwartz argues the opposite: that an excess of choice actually leads to anxiety, paralysis, and a decreased sense of well-being.
---
Score: 278.1050
The first hurdle of "choice overload" is analysis paralysis. When faced with dozens of similar options, the mental effort required to compare them becomes exhausting. Instead of making a "better" choice, many people simply walk away or defer the decision indefinitely. Even when a choice is finally made, a second issue arises: opportunity cost. By picking one path, we become hyper-aware of the attractive features of the paths we rejected. This leads to a lingering sense of regret, even if our chosen option is perfectly functional

#### WAY 3 :

In [17]:
# Using a Retriever, A retriever wraps the vectorstore and plugs into LangChain chains.

retriever = vectorstore.as_retriever(
    search_type="similarity",   # or "mmr" (max marginal relevance)
    search_kwargs={"k": 3}
)

docs = retriever.invoke("According to the essay, does having more choices always make people happier?")

for doc in docs:
    print(doc.page_content)

The Paradox of Choice: When More is Less
In modern consumer societies, we are raised on the mantra that "more is better." We assume that an abundance of options—whether in the grocery aisle, the job market, or on streaming platforms—is the ultimate expression of freedom. However, psychologist Barry Schwartz argues the opposite: that an excess of choice actually leads to anxiety, paralysis, and a decreased sense of well-being.
The first hurdle of "choice overload" is analysis paralysis. When faced with dozens of similar options, the mental effort required to compare them becomes exhausting. Instead of making a "better" choice, many people simply walk away or defer the decision indefinitely. Even when a choice is finally made, a second issue arises: opportunity cost. By picking one path, we become hyper-aware of the attractive features of the paths we rejected. This leads to a lingering sense of regret, even if our chosen option is perfectly functional.
Furthermore, high expectations pla

## SAVING VECTOR DB LOCALLY
WE CAN QUERY DB WITH AND WIHOUT SAVING DB

In [ ]:
# Save the FAISS DB locally
vectorstore.save_local("faiss_index")  # creates a folder named faiss_index/

# Load the FAISS DB from disk
vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True  # required flag in newer LangChain
)
# NOW WE CAN QUERY THSI LOADED FAISS DB FROM DISK AND WE CAN QUERY DB WITH AND WIHOUT SAVING DB

 Retrievers — Deep Explanation
What is a Retriever?
A retriever is a LangChain interface that wraps your vectorstore and turns it into a clean, pluggable component that just does one job — take a query string, return relevant documents.
Think of it like this:
Your Question (string)
        ↓
   [ Retriever ]
        ↓
  Relevant Documents (list)
Without a retriever, you call vectorstore.similarity_search(query) manually every time. With a retriever, you just call retriever.invoke(query) and it works the same — but now it can plug into any LangChain chain, agent, or pipeline automatically.

Why use a Retriever instead of directly querying the vectorstore?
There are 3 big reasons:
1. It's the standard LangChain interface
Every LangChain chain that needs to fetch documents (like RAG chains, QA chains) expects a retriever object, not a raw vectorstore. So if you ever want to build a full pipeline, you need a retriever.
2. You get access to smarter retrieval strategies beyond plain similarity search (explained below).
3. It decouples your retrieval logic from your chain — you can swap FAISS for Chroma or change strategy without touching your chain code.

Types of Retrievers
Type 1 — Similarity Retriever (basic)
The default. Finds the top-k chunks most similar to your query.
pythonretriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)
Problem it has: If your document repeats the same idea in 5 chunks, you'll get back 5 almost identical chunks — wasting your context window.

Type 2 — MMR Retriever (Max Marginal Relevance)
Solves the repetition problem. It fetches a larger pool of candidates first, then selects results that are both relevant AND diverse from each other.
pythonretriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,         # final number of docs to return
        "fetch_k": 20   # candidates to consider before picking diverse 4
    }
)
When to use: When your document has a lot of repetitive or overlapping chunks.

Type 3 — Similarity Score Threshold Retriever
Only returns chunks above a minimum relevance score. Useful when you don't want to return irrelevant results just because k=4 demands 4 results.
pythonretriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "score_threshold": 0.7,  # only return if similarity >= 0.7
        "k": 4
    }
)
When to use: When your query might be completely unrelated to your document and you don't want to return garbage results.

Type 4 — MultiQuery Retriever
This is more advanced. It uses an LLM to automatically generate multiple versions of your query, runs each one, and combines the results. This solves the problem of a poorly worded query missing relevant chunks.
pythonfrom langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_community.llms import Ollama

llm = Ollama(model="llama3")

retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    llm=llm
)

# Internally generates ~3 variations of your question, runs all, deduplicates
docs = retriever.invoke("What does the paper say about transformers?")
When to use: When your users might ask vague or imprecise questions.

Type 5 — Contextual Compression Retriever
Retrieves chunks and then compresses them — extracts only the part of each chunk that actually answers your question, throwing away the noise.
pythonfrom langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain_community.llms import Ollama

llm = Ollama(model="llama3")
compressor = LLMChainExtractor.from_llm(llm)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vectorstore.as_retriever(search_kwargs={"k": 4})
)

docs = compression_retriever.invoke("What is the main finding?")
# Each doc now contains only the relevant sentence(s), not the full chunk
When to use: When your chunks are large and contain lots of off-topic content alongside the relevant part.

Which retriever should you use?
SituationUse thisGetting started / simple use casesimilarityDocument has repetitive/overlapping chunksmmrQuery might not match anything in the docsimilarity_score_thresholdUsers ask vague or imprecise questionsMultiQueryRetrieverChunks are large with mixed contentContextualCompressionRetriever